# Kaggle Notebook: Dataset Preparation for Outfit Transformer
- Loads Polyvore dataset from HuggingFace
- Crops garments using Grounded-SAM (4-level fallback)
- Generates L2-normalized CLIP embeddings (.npy per item)
- Saves per-item metadata JSONs
- Verifies dataset integrity
- Builds FAISS index for retrieval

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

!pip install -q 'git+https://github.com/facebookresearch/sam2.git'
# !pip install -q transformers open_clip_torch datasets faiss-cpu tqdm


In [ ]:
!pip install -q transformers open_clip_torch faiss-cpu tqdm gdown datasets

In [ ]:
import os
import json

# Clone the outfit-transformer repo (same as untitled4.py)
!git clone https://github.com/bigohofone/outfit-transformer.git
os.chdir('outfit-transformer')

# Install repo dependencies
!pip install -q torch torchvision transformers faiss-cpu gdown wandb datasets

# Download pre-trained checkpoints
!mkdir -p checkpoints
!gdown --id 1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi -O checkpoints.zip
!unzip -q -o checkpoints.zip -d ./checkpoints
!rm -f checkpoints.zip
!ls checkpoints/

# ── Load Polyvore dataset from HuggingFace ──
# The outfit-transformer repo's polyvore.py uses 'owj0421/polyvore' (public, no gating).
# This dataset contains: item_id, image (PIL), category, url_name
from datasets import load_dataset
from tqdm.auto import tqdm

print("📥 Loading Polyvore dataset from HuggingFace (owj0421/polyvore)...")
polyvore_ds = load_dataset('owj0421/polyvore')['data']

# Build metadata lookup: item_id -> {category, url_name, image}
item_metadata = {}
for i, item in enumerate(tqdm(polyvore_ds, desc="Building metadata index")):
    item_metadata[item['item_id']] = {
        'index': i,
        'category': item.get('category', ''),
        'url_name': item.get('url_name', ''),
    }

all_item_ids = list(item_metadata.keys())
print(f"✅ Loaded {len(all_item_ids)} items from Polyvore dataset")

# Show a sample entry
sample_id = all_item_ids[0]
print(f"Sample item ({sample_id}): {item_metadata[sample_id]}")
sample_img = polyvore_ds[0]['image']
print(f"Sample image size: {sample_img.size}, mode: {sample_img.mode}")

In [ ]:

import gc
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# Download SAM2 Tiny checkpoint (~40MB, fastest variant)
!mkdir -p checkpoints
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt -O checkpoints/sam2.1_hiera_tiny.pt

sam2_checkpoint = "checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"
d_model_id = "IDEA-Research/grounding-dino-tiny"

# Multi-GPU support: Kaggle provides 2x T4 GPUs.
# We will initialize one SAM service per GPU and run them in parallel threads.
num_gpus = min(2, torch.cuda.device_count()) if torch.cuda.is_available() else 1
print(f"Using {num_gpus} device(s)")

if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


class SAM_service():
    """Grounded-SAM service: Grounding DINO (detection) + SAM2 (segmentation).
    Inlined from store_products.py for Kaggle portability.
    """

    def __init__(self, checkpoint, model_cfg, d_model_id, device):
        self.device = device
        self.sam2_model = build_sam2(model_cfg, checkpoint, device=device)
        self.predictor = SAM2ImagePredictor(self.sam2_model)
        self.processor = AutoProcessor.from_pretrained(d_model_id)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(d_model_id).to(device)
        self._current_image_ref = None

    def _prepare_image(self, image: Image.Image):
        return np.array(image.convert("RGB"))

    def _set_image_if_needed(self, image: Image.Image):
        if self._current_image_ref is not image:
            self.predictor.set_image(image)
            self._current_image_ref = image
            gc.collect()

    def segment_image(self, image: Image.Image, boxes=None, multimask=False):
        self._set_image_if_needed(image)
        masks, scores, logits = self.predictor.predict(box=boxes, multimask_output=multimask)
        gc.collect()
        return masks, scores, logits

    def segment_with_prompt(self, image: Image.Image, prompt: str, multimask=False):
        if isinstance(image, Image.Image):
            if image.mode != "RGB":
                image = image.convert("RGB")
        else:
            raise TypeError("Expected PIL.Image.Image")

        inputs = self.processor(images=image, text=prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]])
        results = self.processor.image_processor.post_process_object_detection(
            outputs, threshold=0.35, target_sizes=target_sizes
        )[0]

        del inputs, outputs
        gc.collect()

        if len(results['boxes']) == 0:
            return [], [], []

        best_box = results['boxes'].cpu().numpy()
        del results

        cleaned_boxes = self.remove_duplicates(best_box, iou_threshold=0.75)
        gc.collect()

        masks, scores, logits = self.segment_image(image, boxes=cleaned_boxes, multimask=multimask)
        return masks, cleaned_boxes, scores

    def calculate_iou(self, box, boxes):
        x1 = np.maximum(box[0], boxes[:, 0])
        y1 = np.maximum(box[1], boxes[:, 1])
        x2 = np.minimum(box[2], boxes[:, 2])
        y2 = np.minimum(box[3], boxes[:, 3])
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        box_area = (box[2] - box[0]) * (box[3] - box[1])
        boxes_area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        union_area = box_area + boxes_area - intersection_area
        return intersection_area / (union_area + 1e-6)

    def remove_duplicates(self, boxes, iou_threshold=0.85):
        if len(boxes) == 0:
            return []
        indices = np.arange(len(boxes))
        keep = []
        while len(indices) > 0:
            current_idx = indices[0]
            keep.append(current_idx)
            remaining_indices = indices[1:]
            current_box = boxes[current_idx]
            remaining_boxes = boxes[remaining_indices]
            ious = self.calculate_iou(current_box, remaining_boxes)
            under_threshold_indices = np.where(ious < iou_threshold)[0]
            indices = remaining_indices[under_threshold_indices]
        return boxes[keep]

    def get_segmented_image(self, image: Image.Image, mask):
        """Applies mask, crops to bounding box, returns transparent PNG."""
        image_np = self._prepare_image(image)
        if len(mask.shape) > 2:
            mask = mask[0]
        h, w, c = image_np.shape
        rgba_image = np.zeros((h, w, 4), dtype=np.uint8)
        rgba_image[..., :3] = image_np
        binary_mask = mask > 0
        rgba_image[..., 3] = np.where(binary_mask, 255, 0)

        y_indices, x_indices = np.where(binary_mask)
        if len(y_indices) == 0 or len(x_indices) == 0:
            return Image.fromarray(rgba_image)

        top, bottom = y_indices.min(), y_indices.max()
        left, right = x_indices.min(), x_indices.max()
        cropped_rgba = rgba_image[top:bottom+1, left:right+1]
        return Image.fromarray(cropped_rgba)


sams = []
for i in range(num_gpus):
    dev = torch.device(f"cuda:{i}" if torch.cuda.is_available() else "cpu")
    print(f"Initializing SAM service on {dev}...")
    sams.append(SAM_service(sam2_checkpoint, model_cfg, d_model_id, device=dev))
print(f"✅ {len(sams)} SAM service(s) initialized (SAM2 Tiny + Grounding DINO)")

In [ ]:
import os

def print_tree(path):
    for root, dirs, files in os.walk(path):
        # Calculate depth for indentation
        level = root.replace(path, '').count(os.sep)
        indent = ' ' * 4 * level
        
        # Get the directory name and the count of files inside it
        dir_name = os.path.basename(root) or path
        file_count = len(files)
        
        print(f'{indent}{dir_name}/ ({file_count} files)')
        
        # Print individual files
        sub_indent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{sub_indent}{f}')

# Start from current directory
print_tree('.')


In [ ]:
import json
import traceback
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# Output directories
OUTPUT_DIR = Path("/kaggle/working/cropped_polyvore")
IMAGES_DIR = OUTPUT_DIR / "images"
METADATA_DIR = OUTPUT_DIR / "metadata"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)


def build_prompt(item_id, metadata_dict):
    """Build Grounding DINO prompt from Polyvore metadata.
    Priority: url_name (description) > category > fallback.
    Fields from owj0421/polyvore: item_id, image, category, url_name.
    """
    meta = metadata_dict.get(item_id, {})

    url_name = meta.get("url_name", "")
    if url_name and isinstance(url_name, str) and len(url_name.strip()) > 2:
        prompt = url_name.strip().replace("-", " ").replace("_", " ")
    else:
        cat = meta.get("category", "")
        if cat and isinstance(cat, str) and len(cat.strip()) > 1:
            prompt = cat.strip()
        else:
            prompt = "clothing"

    if not prompt.endswith("."):
        prompt += "."
    return prompt.lower()


def crop_garment_fast(image, prompt, sam_service):
    """
    Fast garment crop using DINO bbox only (no SAM segmentation).
    Polyvore images are clean product photos — bbox crop is sufficient.

    Fallback hierarchy:
      1. Grounding DINO bbox crop  — detect garment, crop to bounding box
      2. Center crop               — if DINO finds nothing, center-crop to square
      3. Original resized          — last resort
    """
    meta = {
        "prompt": prompt,
        "method": None,
        "bbox": None,
        "confidence": None,
        "failure_reason": None,
        "original_size": list(image.size),
    }

    if image.mode != "RGB":
        image = image.convert("RGB")

    # ── Level 1: Grounding DINO bounding-box crop (fp16) ──
    try:
        inputs = sam_service.processor(
            images=image, text=prompt, return_tensors="pt"
        ).to(sam_service.device)

        with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=sam_service.device.type == "cuda"):
            outputs = sam_service.model(**inputs)

        results = sam_service.processor.image_processor.post_process_object_detection(
            outputs, threshold=0.25,
            target_sizes=torch.tensor([image.size[::-1]])
        )[0]
        del inputs, outputs

        if len(results['boxes']) > 0:
            box = results['boxes'][0].cpu().numpy().astype(int)
            w, h = image.size
            box = [max(0, box[0]), max(0, box[1]), min(w, box[2]), min(h, box[3])]
            # Sanity: skip tiny boxes (< 5% of image area)
            box_area = (box[2] - box[0]) * (box[3] - box[1])
            if box_area > 0.05 * w * h:
                cropped = image.crop(tuple(box))
                meta["method"] = "bbox"
                meta["bbox"] = [int(b) for b in box]
                meta["confidence"] = float(np.asarray(results['scores'][0].cpu()).item())
                meta["cropped_size"] = list(cropped.size)
                del results
                return cropped, meta
        del results
    except Exception as e:
        meta["failure_reason"] = f"DINO bbox failed: {str(e)[:200]}"

    # ── Level 2: Center crop ──
    try:
        w, h = image.size
        size = min(w, h)
        left = (w - size) // 2
        top = (h - size) // 2
        cropped = image.crop((left, top, left + size, top + size))
        meta["method"] = "center"
        meta["cropped_size"] = list(cropped.size)
        return cropped, meta
    except Exception as e:
        meta["failure_reason"] = f"Center crop failed: {str(e)[:200]}"

    # ── Level 3: Original ──
    meta["method"] = "original"
    meta["failure_reason"] = meta.get("failure_reason", "All methods failed")
    meta["cropped_size"] = list(image.size)
    return image.copy(), meta


def process_items_chunk(item_ids, metadata_dict, dataset, sam_service, worker_id):
    """Process a chunk of items on a single GPU."""
    succeeded = 0
    failed_items = []
    method_counts = {"bbox": 0, "center": 0, "original": 0}

    disable_pbar = (worker_id > 0)
    pbar = tqdm(item_ids, desc=f"📦 GPU:{sam_service.device.index} Worker {worker_id}",
                unit="item", disable=disable_pbar)

    for item_id in pbar:
        out_img_path = IMAGES_DIR / f"{item_id}.png"
        meta_path = METADATA_DIR / f"{item_id}.json"

        # Resume support: skip already processed
        if out_img_path.exists() and meta_path.exists():
            succeeded += 1
            continue

        try:
            ds_index = metadata_dict[item_id]['index']
            image = dataset[ds_index]['image']
            if not isinstance(image, Image.Image):
                failed_items.append({"item_id": item_id, "reason": "invalid image"})
                continue
            image = image.convert("RGB")

            prompt = build_prompt(item_id, metadata_dict)
            cropped, meta = crop_garment_fast(image, prompt, sam_service)

            cropped.save(str(out_img_path), "PNG")

            meta["item_id"] = item_id
            meta["timestamp"] = datetime.now().isoformat()
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)

            method_counts[meta["method"]] = method_counts.get(meta["method"], 0) + 1
            succeeded += 1

        except Exception as e:
            failed_items.append({"item_id": item_id, "reason": str(e)[:300]})
            traceback.print_exc()

        if not disable_pbar:
            pbar.set_postfix(ok=succeeded, fail=len(failed_items))

        # Free memory every 500 items (not every 50 — gc.collect is expensive)
        if succeeded % 500 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return succeeded, failed_items, method_counts


# ── Process all items using ThreadPoolExecutor ──
from concurrent.futures import ThreadPoolExecutor

print(f"\n{'='*50}")
print(f"Processing {len(all_item_ids)} items with {num_gpus} parallel worker(s)")
print(f"{'='*50}")

chunks = np.array_split(all_item_ids, num_gpus)

total_succeeded = 0
all_failed = []
all_methods = {}

with ThreadPoolExecutor(max_workers=num_gpus) as executor:
    futures = []
    for i in range(num_gpus):
        chunk = chunks[i].tolist()
        futures.append(
            executor.submit(process_items_chunk, chunk, item_metadata, polyvore_ds, sams[i], i)
        )

    for f in futures:
        ok, fails, methods = f.result()
        total_succeeded += ok
        all_failed.extend(fails)
        for k, v in methods.items():
            all_methods[k] = all_methods.get(k, 0) + v

if all_failed:
    with open(OUTPUT_DIR / "failed_items.json", "w") as f:
        json.dump(all_failed, f, indent=2)

print(f"\n🎉 Total: {total_succeeded} cropped, {len(all_failed)} failed")
print(f"Methods breakdown: {all_methods}")

## embeddings_raw/
    Original CLIP embeddings without normalization.

## embeddings_normalized/
    L2-normalized embeddings for:
    - cosine similarity
    - FAISS retrieval
    - outfit recommendation
    - compatibility learning

## Benefits of saving both:
    - Research flexibility
    - Future fine-tuning
    - Better debugging
    - Easier ablation experiments

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import open_clip

# ============================================================
# Directories
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/cropped_polyvore")

IMAGES_DIR = OUTPUT_DIR / "images"

RAW_EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings_raw"
NORM_EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings_normalized"

RAW_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
NORM_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Device
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ============================================================
# Load FashionCLIP / OpenCLIP
# ============================================================

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)

clip_model = clip_model.to(device)
clip_model.eval()

print("✅ CLIP model loaded")

# ============================================================
# Free SAM memory (if SAM was used previously)
# ============================================================

try:
    del sam
except:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🧹 SAM service freed from GPU memory")

# ============================================================
# Embedding Generation
# ============================================================

def generate_embedding(image_path, clip_model, preprocess, device):
    """
    Generate both raw and L2-normalized CLIP embeddings.

    Returns:
        raw_emb  : np.ndarray(shape=(512,), dtype=float32)
        norm_emb : np.ndarray(shape=(512,), dtype=float32)
    """

    image = Image.open(image_path).convert("RGB")

    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = clip_model.encode_image(image_tensor)

    # Convert to numpy
    raw_emb = embedding.cpu().numpy().flatten().astype(np.float32)

    # Create normalized embedding
    norm_emb = raw_emb.copy()

    norm = np.linalg.norm(norm_emb)

    if norm > 0:
        norm_emb = norm_emb / norm

    return raw_emb, norm_emb


# ============================================================
# Process Images
# ============================================================

image_files = sorted(IMAGES_DIR.glob("*.png"))

print(f"Generating embeddings for {len(image_files)} images...")

embed_errors = []

for img_path in tqdm(image_files, desc="🔢 Embeddings", unit="img"):

    item_id = img_path.stem

    raw_path = RAW_EMBEDDINGS_DIR / f"{item_id}.npy"
    norm_path = NORM_EMBEDDINGS_DIR / f"{item_id}.npy"

    # --------------------------------------------------------
    # Resume Support
    # --------------------------------------------------------

    if raw_path.exists() and norm_path.exists():
        continue

    try:

        raw_emb, norm_emb = generate_embedding(
            img_path,
            clip_model,
            clip_preprocess,
            device
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        if raw_emb.shape[0] != 512:
            raise ValueError(
                f"Invalid embedding shape: {raw_emb.shape}"
            )

        if np.isnan(raw_emb).any():
            raise ValueError("NaN detected in raw embedding")

        if np.isnan(norm_emb).any():
            raise ValueError("NaN detected in normalized embedding")

        # ----------------------------------------------------
        # Save embeddings
        # ----------------------------------------------------

        np.save(raw_path, raw_emb)
        np.save(norm_path, norm_emb)

    except Exception as e:

        embed_errors.append({
            "item_id": item_id,
            "reason": str(e)[:300]
        })

# ============================================================
# Save Errors
# ============================================================

if embed_errors:

    print(f"⚠ {len(embed_errors)} embedding errors")

    with open(OUTPUT_DIR / "embedding_errors.json", "w") as f:
        json.dump(embed_errors, f, indent=2)

# ============================================================
# Final Summary
# ============================================================

raw_count = len(list(RAW_EMBEDDINGS_DIR.glob("*.npy")))
norm_count = len(list(NORM_EMBEDDINGS_DIR.glob("*.npy")))

print("\n==================== SUMMARY ====================")
print(f"✅ Raw embeddings saved:        {raw_count}")
print(f"✅ Normalized embeddings saved: {norm_count}")
print(f"⚠ Errors:                      {len(embed_errors)}")
print("=================================================\n")


In [ ]:
def verify_dataset(images_dir, embeddings_dir, metadata_dir):
    """
    Verify:
    - image exists
    - image can be opened
    - crop is non-empty (width > 0, height > 0)
    - embedding shape == (512,)
    - no NaNs in embedding
    """
    errors = []
    stats = {
        "total": 0, "valid": 0, "invalid": 0,
        "methods": {"sam": 0, "bbox": 0, "center": 0, "original": 0, "unknown": 0}
    }

    npy_files = sorted(Path(embeddings_dir).glob("*.npy"))
    print(f"Verifying {len(npy_files)} items...")

    for npy_file in tqdm(npy_files, desc="🔍 Verifying", unit="item"):
        item_id = npy_file.stem
        stats["total"] += 1

        img_path = Path(images_dir) / f"{item_id}.png"
        meta_path = Path(metadata_dir) / f"{item_id}.json"

        # 1. Image exists
        if not img_path.exists():
            errors.append((item_id, "image missing"))
            stats["invalid"] += 1
            continue

        # 2. Image can be opened
        try:
            img = Image.open(img_path)
            w, h = img.size
        except Exception as e:
            errors.append((item_id, f"image corrupt: {e}"))
            stats["invalid"] += 1
            continue

        # 3. Crop is non-empty
        if w == 0 or h == 0:
            errors.append((item_id, "empty crop (0 dimension)"))
            stats["invalid"] += 1
            continue

        # 4. Embedding shape
        emb = np.load(npy_file)
        if emb.shape != (512,):
            errors.append((item_id, f"bad embedding shape: {emb.shape}"))
            stats["invalid"] += 1
            continue

        # 5. No NaNs
        if np.any(np.isnan(emb)):
            errors.append((item_id, "NaN in embedding"))
            stats["invalid"] += 1
            continue

        stats["valid"] += 1

        # Count crop methods from metadata
        if meta_path.exists():
            try:
                with open(meta_path) as f:
                    meta = json.load(f)
                method = meta.get("method", "unknown")
                stats["methods"][method] = stats["methods"].get(method, 0) + 1
            except Exception:
                stats["methods"]["unknown"] += 1

    return stats, errors


stats, errors = verify_dataset(IMAGES_DIR, EMBEDDINGS_DIR, METADATA_DIR)

print(f"\n{'='*50}")
print(f"📊 DATASET VERIFICATION RESULTS")
print(f"{'='*50}")
print(f"Total:   {stats['total']}")
print(f"Valid:   {stats['valid']} ✅")
print(f"Invalid: {stats['invalid']} ❌")
print(f"Methods: {stats['methods']}")

if errors:
    print(f"\n⚠ {len(errors)} errors found:")
    for item_id, reason in errors[:20]:
        print(f"  - {item_id}: {reason}")
    if len(errors) > 20:
        print(f"  ... and {len(errors) - 20} more")

In [ ]:
import faiss

def build_faiss_index(embeddings_dir, output_dir):
    """Build FAISS IndexFlatIP from L2-normalized embeddings."""
    item_ids = []
    embeddings = []

    npy_files = sorted(Path(embeddings_dir).glob("*.npy"))
    for npy_file in tqdm(npy_files, desc="📦 Building FAISS index"):
        emb = np.load(npy_file).astype(np.float32)
        if emb.shape == (512,) and not np.any(np.isnan(emb)):
            item_ids.append(npy_file.stem)
            embeddings.append(emb)

    embeddings_matrix = np.stack(embeddings)  # (N, 512)

    # IndexFlatIP = inner product (= cosine similarity when vectors are L2-normalized)
    index = faiss.IndexFlatIP(512)
    index.add(embeddings_matrix)

    output_path = Path(output_dir)
    faiss.write_index(index, str(output_path / "faiss_index.bin"))

    with open(output_path / "faiss_index.json", "w") as f:
        json.dump(item_ids, f)

    print(f"✅ FAISS index built: {index.ntotal} vectors")
    return index, item_ids


faiss_index, faiss_ids = build_faiss_index(EMBEDDINGS_DIR, OUTPUT_DIR)

In [ ]:

# Save preparation summary
summary = {
    "dataset": "polyvore (outfit-transformer repo)",
    "seed": seed,
    "sam_model": "sam2.1_hiera_tiny",
    "clip_model": "ViT-B-32 (laion2b_s34b_b79k)",
    "total_items_in_metadata": len(item_metadata),
    "total_items_with_images": len(all_item_ids),
    "total_images_cropped": stats["valid"],
    "total_failed": stats["invalid"],
    "crop_methods": stats["methods"],
    "embedding_dim": 512,
    "faiss_vectors": faiss_index.ntotal,
    "timestamp": datetime.now().isoformat(),
}

with open(OUTPUT_DIR / "preparation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 50)
print("📊 PREPARATION SUMMARY")
print("=" * 50)
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
!pip install -q kaggle
kaggle_meta = {
    "title": "polyvore-disjoint-cropped-garments",
    "id": "mabdelmoneim/polyvore-disjoint-cropped-garments",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
    json.dump(kaggle_meta, f)
!kaggle datasets create -p {OUTPUT_DIR} --dir-mode zip

In [ ]:
print("\n🎉 Dataset preparation complete!")
print(f"Output directory: {OUTPUT_DIR}")
print("To create a Kaggle Dataset: Go to Output tab → New Dataset")